In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# 📊 Analyse Exploratoire - German Credit Dataset\n",
    "\n",
    "Notebook d'analyse exploratoire pour le Credit Scoring Model.\n",
    "\n",
    "**Projet** : CodeAlpha Machine Learning Internship  \n",
    "**Dataset** : German Credit (UCI ML Repository)  \n",
    "**Objectif** : Comprendre les données avant la modélisation"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Imports\n",
    "import pandas as pd\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "\n",
    "plt.style.use('seaborn-v0_8-whitegrid')\n",
    "sns.set_palette(\"husl\")\n",
    "\n",
    "# Chargement des données\n",
    "df = pd.read_csv('../data/german_credit.csv')\n",
    "print(f\"Shape: {df.shape}\")\n",
    "df.head(10)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 1. Aperçu du dataset"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Informations générales\n",
    "print(\"Types de données:\")\n",
    "print(df.dtypes)\n",
    "print(\"\\nValeurs manquantes:\")\n",
    "print(df.isnull().sum())\n",
    "print(\"\\nDoublons:\", df.duplicated().sum())"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 2. Distribution de la cible"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Distribution de la cible\n",
    "fig, ax = plt.subplots(figsize=(8, 6))\n",
    "target_counts = df['target'].value_counts()\n",
    "colors = ['#2ecc71', '#e74c3c']\n",
    "labels = ['Bon Crédit (0)', 'Mauvais Crédit (1)']\n",
    "wedges, texts, autotexts = ax.pie(target_counts, labels=labels, autopct='%1.1f%%', \n",
    "                                   colors=colors, startangle=90, explode=(0, 0.05))\n",
    "ax.set_title('Distribution de la Cible', fontsize=14, fontweight='bold')\n",
    "plt.tight_layout()\n",
    "plt.savefig('../results/target_distribution.png', dpi=150)\n",
    "plt.show()\n",
    "\n",
    "print(f\"\\nBon crédit: {target_counts[0]} ({target_counts[0]/len(df)*100:.1f}%)\")\n",
    "print(f\"Mauvais crédit: {target_counts[1]} ({target_counts[1]/len(df)*100:.1f}%)\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 3. Statistiques descriptives"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Variables numériques\n",
    "numeric_cols = ['duration', 'credit_amount', 'installment_rate', \n",
    "                'residence_since', 'age', 'num_credits', 'num_dependents']\n",
    "\n",
    "print(\"Statistiques descriptives:\")\n",
    "df[numeric_cols].describe().round(2)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Distribution des variables numériques\n",
    "fig, axes = plt.subplots(2, 4, figsize=(16, 8))\n",
    "axes = axes.flatten()\n",
    "\n",
    "for idx, col in enumerate(numeric_cols):\n    ax = axes[idx]\n",
    "    df[col].hist(bins=20, ax=ax, color='steelblue', edgecolor='white', alpha=0.7)\n",
    "    ax.set_title(f'Distribution de {col}')\n    ax.set_xlabel(col)\n    ax.set_ylabel('Fréquence')\n",
    "\n",
    "axes[-1].axis('off')\n",
    "plt.tight_layout()\n",
    "plt.savefig('../results/numeric_distributions.png', dpi=150)\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 4. Analyse par cible"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Montant du crédit par cible\n",
    "fig, axes = plt.subplots(1, 2, figsize=(14, 5))\n",
    "\n",
    "sns.boxplot(data=df, x='target', y='credit_amount', ax=axes[0], \n",
    "            palette=['#2ecc71', '#e74c3c'])\n",
    "axes[0].set_xticklabels(['Bon Crédit', 'Mauvais Crédit'])\n",
    "axes[0].set_title('Montant du Crédit par Cible')\n",
    "\n",
    "sns.violinplot(data=df, x='target', y='duration', ax=axes[1],\n",
    "               palette=['#2ecc71', '#e74c3c'])\n",
    "axes[1].set_xticklabels(['Bon Crédit', 'Mauvais Crédit'])\n",
    "axes[1].set_title('Durée du Crédit par Cible')\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.savefig('../results/target_analysis.png', dpi=150)\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 5. Matrice de corrélation"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Correlation matrix\n",
    "numeric_with_target = numeric_cols + ['target']\n",
    "corr = df[numeric_with_target].corr()\n",
    "\n",
    "fig, ax = plt.subplots(figsize=(10, 8))\n",
    "sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, \n",
    "            square=True, linewidths=0.5, ax=ax)\n",
    "ax.set_title('Matrice de Corrélation', fontsize=14, fontweight='bold')\n",
    "plt.tight_layout()\n",
    "plt.savefig('../results/correlation_matrix.png', dpi=150)\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 6. Variables catégorielles"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Variables catégorielles\n",
    "categorical_cols = df.select_dtypes(include=['object']).columns.tolist()\n",
    "\n",
    "fig, axes = plt.subplots(3, 5, figsize=(20, 12))\n",
    "axes = axes.flatten()\n",
    "\n",
    "for idx, col in enumerate(categorical_cols):\n    ax = axes[idx]\n",
    "    df[col].value_counts().plot(kind='bar', ax=ax, color='steelblue', alpha=0.7)\n",
    "    ax.set_title(f'Distribution de {col}')\n",
    "    ax.set_xlabel('')\n",
    "    ax.tick_params(axis='x', rotation=45)\n",
    "\n",
    "for idx in range(len(categorical_cols), len(axes)):\n    axes[idx].axis('off')\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.savefig('../results/categorical_distributions.png', dpi=150)\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 7. Age vs Montant du crédit"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "fig, ax = plt.subplots(figsize=(10, 6))\n",
    "scatter = ax.scatter(df['age'], df['credit_amount'], c=df['target'], \n",
    "                     cmap='RdYlGn_r', alpha=0.6, s=50)\n",
    "ax.set_xlabel('Âge')\n",
    "ax.set_ylabel('Montant du Crédit')\n",
    "ax.set_title('Age vs Montant du Crédit (coloré par cible)')\n",
    "plt.colorbar(scatter, label='0=Bon, 1=Mauvais')\n",
    "plt.tight_layout()\n",
    "plt.savefig('../results/age_vs_credit.png', dpi=150)\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 8. Résumé"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "print(\"=\" * 50)\n",
    "print(\"RESUME DE L'ANALYSE EXPLORATOIRE\")\n",
    "print(\"=\" * 50)\n",
    "print(f\"\\nDataset: {df.shape[0]} lignes, {df.shape[1]} colonnes\")\n",
    "print(f\"Cible: {target_counts[0]} bon crédit, {target_counts[1]} mauvais crédit\")\n",
    "print(f\"Features numériques: {len(numeric_cols)}\")\n",
    "print(f\"Features catégorielles: {len(categorical_cols)}\")\n",
    "print(f\"\\nVariables catégorielles: {categorical_cols}\")\n",
    "print(\"\\n✅ Analyse exploratoire terminée!\")\n",
    "print(\"📁 Graphiques sauvegardés dans /results\")"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.11.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}